# Fingerprint processing
This is a GFAS PbP processing routine where the PbP data format is contained but the datasets are being modified and saved for further use.

It uses the following datasets: 
- raw PbP observational data: the cloud event 11.-12.08.2024 and dust event 16.08.2024
- scale factors in PbP_processing/detector_scaling_factors

Input needed:
    - path of the data

Sub-routines needed:
- PbP_processing/GFAS101_processing
- GFAS_reading/GFAS101_read_in
- PbP_processing/loss_Calc_functions_GFAS101

Processing applied
- attenuation
- scaling towards theoretical values
- sizing
- losses
- diameter binning

Output: corrected datasets of
- Laboratory measurements 
- cloud_fingerprint: cloud event r_pol and r_F/B dataset
- dust_fingerprint: dust event r_pol and r_F/B dataset


author: Lea Haberstock, Stockholm University, Department of Environmental Science, Atmospheric Unit \
developed toether with: Darrel Baumgardner and Paul Zieger

contact: lea.haberstock@aces.su.se \
used in publication: Haberstock et al. 2026 (submitted to AMT) 

In [3]:
import sys  
import numpy as np
import pandas as pd
import glob2
import glob, os
import os
from io import BytesIO
import datetime as dt
import calendar
from pandas.errors import EmptyDataError
from scipy.ndimage import shift
import scipy
import metpy
from metpy.calc import wind_components,wind_speed,wind_direction
from metpy.units import units
from matplotlib.pyplot import figure
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as colors
import shapely 
import matplotlib.ticker as ticker
import matplotlib 
import PyMieScatt  as pms
from scipy.stats import linregress
import dask
import dask.dataframe as dd
from pathlib import Path

project_dir = Path.cwd().parent
sys.path.append(str(project_dir))
from PbP_processing.GFAS101_processing import *
from calibrations.MieCalculations import *
from GFAS_reading.GFAS101_read_in import *
'''
from GFAS101_processing import *
from MieCalculations import *
from GFAS101_read_in import *'''

'\nfrom GFAS101_processing import *\nfrom MieCalculations import *\nfrom GFAS101_read_in import *'

In [5]:
# change to your work directory
#os.chdir('')
path = '../PbP_processing/detector_scaling_factors'
scales = (pd.read_csv(path, index_col = 0)).squeeze()

In [6]:
# bins for binning the Sizer Peak and Diameter
bins = np.logspace(np.log10(1e-7), np.log10(5.357364e-06), 31)
bi = np.array([0,3,5,9,14,21,50])

In [15]:
path = '/static/StationData/Campaigns/CIS_Sonnblick2024/GFAS'
starttime = '20240811'
endtime = '20240812'
timefmt = '%Y%m%d'

cloud1 = readPbP101(path, starttime, starttime, timefmt, filetype='GFAS_PbP_Data', compute=False)
cloud1 = cloud1[cloud1['datetime'] >= pd.to_datetime('2024-08-11 21:36:00')]
cloud2 = readPbP101(path, endtime, endtime, timefmt, filetype='GFAS_PbP_Data', compute=False)
cloud2 = cloud2[cloud2['datetime'] <= pd.to_datetime('2024-08-12 00:35:00')]

cloud = dd.concat([cloud1, cloud2], ignore_index=True)
cloud = cloud.map_partitions(attenuation)
cloud = cloud.map_partitions(qual_filter)

cloud['scaled Sizer Peak'] = cloud['Sizer Peak'] * scales['forward']
cloud['scaled S'] = cloud['S Peak'] * scales['S']
cloud['scaled P'] = cloud['P Peak'] * scales['P']
cloud['r_pol'] = (cloud['scaled S'] - cloud['scaled P']) / (cloud['scaled S'] + cloud['scaled P'])
cloud['F/B'] = cloud['scaled Sizer Peak']/(cloud['scaled S'] + cloud['scaled P'])

cloud['binned Sizer Peak'] = cloud['scaled Sizer Peak'].map_partitions(pd.cut, bins=bins, labels=bins[1:], include_lowest=True, meta=pd.Series(dtype="category"))

meta = sizing_model(cloud._meta.copy())
cloud = cloud.map_partitions(sizing_model, meta=meta)
cloud['binned Diameter'] = cloud['Diameter'].map_partitions(pd.cut, bins = bi, labels  = bi[1:], include_lowest = True)
cloud = cloud.compute()


Scheduled 436 CSV and 0 ZIP files.
Scheduled 282 CSV and 0 ZIP files.


In [16]:
path = '/static/StationData/Campaigns/CIS_Sonnblick2024/GFAS'
starttime = '20240816'
endtime = '20240816'
timefmt = '%Y%m%d'

dust = readPbP101(path, starttime, endtime, timefmt, filetype='GFAS_PbP_Data', compute=False)

Scheduled 444 CSV and 0 ZIP files.


In [ ]:
cloud.loc[:,['r_pol','F/B','binned Diameter','Diameter']].to_csv(f'/share/Lea/data/Paper1/GFAS_field/fingerprint_cloud.csv', index=False)

In [17]:
dust = dust[(dust['datetime'] >= pd.to_datetime('2024-08-16 03:00'))& (dust['datetime'] <= pd.to_datetime('2024-08-16 12:00'))]

dust = dust.map_partitions(attenuation)
dust = dust.map_partitions(qual_filter)
dust['scaled Sizer Peak'] = dust['Sizer Peak'] * scales['forward']
dust['scaled S'] = dust['S Peak'] * scales['S']
dust['scaled P'] = dust['P Peak'] * scales['P']
dust['r_pol'] = (dust['scaled S'] - dust['scaled P']) / (dust['scaled S'] + dust['scaled P'])
dust['F/B'] = dust['scaled Sizer Peak']/(dust['scaled S'] + dust['scaled P'])

bins = np.logspace(np.log10(1e-7), np.log10(5.357364e-06), 31)
dust['binned Sizer Peak'] = dust['scaled Sizer Peak'].map_partitions(pd.cut, bins=bins, labels=bins[1:], include_lowest=True, meta=pd.Series(dtype="category"))

meta = sizing_model(dust._meta.copy())
dust = dust.map_partitions(sizing_model, meta=meta)
dust['binned Diameter'] = dust['Diameter'].map_partitions(pd.cut, bins = bi, labels  = bi[1:], include_lowest = True)
dust = dust.compute()

In [ ]:
dust.loc[:,['r_pol','F/B','binned Diameter','Diameter']].to_csv(f'/share/Lea/data/Paper1/GFAS_field/fingerprint_dust.csv', index=False)

In [7]:
# add lab experiments
dustIceland  = pd.read_csv('/share/Lea/data/LabExperiments2024/dustIceland.csv', index_col=0)
DustSvalbard  = pd.read_csv('/share/Lea/data/LabExperiments2024/DustSvalbard.csv', index_col=0)
volcano  = pd.read_csv('/share/Lea/data/LabExperiments2024/volcano.csv', index_col=0)
cellulose  = pd.read_csv('/share/Lea/data/LabExperiments2024/cellulose.csv', index_col=0)
glassbeads15  = pd.read_csv('/share/Lea/data/LabExperiments2024/glassbeads15.csv', index_col=0)
water  = pd.read_csv('/share/Lea/data/LabExperiments2024/maxi.csv', index_col=0)

df = [volcano, cellulose, glassbeads15, water] # DustSvalbard, dustIceland,

for D in df:
    qual_filter(D)
    attenuation(D)
    sizing_model(D)
    D['scaled Sizer Peak'] = D['Sizer Peak'] * scales['forward']
    D['scaled S'] = D['S Peak'] * scales['S']
    D['scaled P'] = D['P Peak'] * scales['P']
    D['r_pol'] = (D['scaled S'] - D['scaled P']) / (D['scaled S'] + D['scaled P'])
    D['F/B'] = D['scaled Sizer Peak']/(D['scaled S'] + D['scaled P'])
    D['binned Sizer Peak'] = pd.cut(D['scaled Sizer Peak'], bins=bins, labels=bins[1:], include_lowest=True)
    D['binned Diameter'] = pd.cut(D['Diameter'], bins = bi, labels  = bi[1:], include_lowest = True)
    D.drop(columns='Mid Bin Size (µm)', inplace=True)
    try:
        D.drop(columns = 'mid_size', inplace = True)
    except KeyError:
        pass

volcano.to_csv(f'/share/Lea/data/Paper1/GFAS_laboratory/volcano.csv', index=False)
cellulose.to_csv(f'/share/Lea/data/Paper1/GFAS_laboratory/cellulose.csv', index=False)
glassbeads15.to_csv(f'/share/Lea/data/Paper1/GFAS_laboratory/gb15.csv', index=False)
water.to_csv(f'/share/Lea/data/Paper1/GFAS_laboratory/water.csv', index=False)
